<a href="https://colab.research.google.com/github/LuckyBoy721/Midterm-test-graph-data/blob/main/UTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEMBAR JAWABAN UJIAN TENGAH SEMESTER (UTS)
### MATAKULIAH: GRAF SAINS DATA

**BIODATA MAHASISWA**
* **Nama:** Muhammad Radifan Asyauri
* **NIM:** 23031554114
* **Program Studi:** S1 Sains Data
* **Fakultas:** Fakultas Matematika dan Ilmu Pengetahuan Alam (FMIPA)
* **Universitas:** Universitas Negeri Surabaya (UNESA)

---

## INTERPRETASI DAN ANALISIS HASIL JAWABAN UTS

### 2.1 Wrangling dan EDA

1. **Pembagian Periode Waktu:**
   * **P1 (Training):** `2015-05-20` s.d `2015-07-20` (Data latih utama dengan total interaksi sebanyak 1.259.732 records).
   * **P2 (Testing):** `2015-07-29` s.d `2015-08-13` (Data uji masa depan untuk evaluasi dengan total interaksi sebanyak 279.067 records).

2. **Komposisi Subgraph (Subset 1000 User Acak):**
   * Jaringan interaksi bipartite yang terbentuk terdiri atas **1.000 User unik** dan **1.301 Item unik** dengan total **1.709 relasi interaksi**.
   * *Cypher Query Neo4j* untuk memeriksa statistik ini secara langsung:
     ```cypher
     MATCH (n) RETURN labels(n) AS Label, count(n) AS Jumlah
     ```

3. **Karakteristik Vector Space Modelling (VSM):**
   * **VSM User (Matriks R):** Berdimensi `(817, 1064)`. Dimensi ini mewakili 817 user aktif di P1 terhadap 1.064 item unik.
   * **VSM Item (Matriks R.T):** Berdimensi `(1064, 817)` sebagai representasi setiap item berdasarkan karakteristik user yang berinteraksi dengannya.

4. **Analisis Proyeksi Graf Terbaik:**
   * Proyeksi **Item-Item** terbukti jauh lebih efektif dan efisien secara komputasional dibandingkan proyeksi User-User. Jumlah item dalam ekosistem retail cenderung lebih statis dan memiliki pola perilaku yang konsisten, sedangkan jumlah user baru bertambah secara eksponensial di dunia nyata yang dapat membuat proyeksi User-User cepat tidak relevan.

5. **Analisis Similarity Graf (Sebelum vs Sesudah):**
   * Perbandingan graf menunjukkan transisi dari graf bipartite (User terhubung ke Item melalui tepi interaksi) ke graf proyeksi homogen di mana simpul User terhubung langsung dengan User lain berdasarkan nilai kesamaan *Cosine Similarity*.

---

### 2.2 Analytics (Rekomendasi & Evaluasi)

1. **Hasil Rekomendasi KNN ($k=5$):**
   * **Target User:** ID `168204`
   * **5 Tetangga Terdekat:** `[1288975, 479094, 479055, 477078, 476278]`
   * **3 Item Rekomendasi Teratas:** `[133256, 113801, 368719]`

2. **Metrik Evaluasi:**
   * **Hit@5:** `0`
   * **NDCG@5:** `0.0000`
   * **Interpretasi Nilai Nol:** Nilai metrik evaluasi bernilai nol dikarenakan keterbatasan data pengujian ($P_2$) pada user sampel yang bersifat sangat renggang (*sparse*). Hal ini menunjukkan perlunya strategi penanganan kekosongan data (*data sparsity*) di masa mendatang.

In [ ]:
with open('/content/naskah_soal_uts_graf_sains_data.md', 'r') as f:
    print(f.read())

In [ ]:
import re
with open('/content/naskah_soal_uts_graf_sains_data.md', 'r') as f:
    content = f.read()
# Find curl or wget commands
urls = re.findall(r'(curl[^\n]+|wget[^\n]+|https?://[^\s"\)\]]+)', content)
for url in urls:
    print(url)

In [ ]:
!curl -L -o ecommerce-dataset.zip https://www.kaggle.com/api/v1/datasets/download/retailrocket/ecommerce-dataset
!unzip -o ecommerce-dataset.zip
!ls -la

In [ ]:
import pandas as pd
import datetime

# Membaca data interaksi untuk mengecek rentang waktu
events_df = pd.read_csv('/content/events.csv')
print("Jumlah total baris:", len(events_df))

# Mengonversi timestamp (milidetik) ke format datetime
events_df['datetime'] = pd.to_datetime(events_df['timestamp'], unit='ms')
min_date = events_df['datetime'].min()
max_date = events_df['datetime'].max()

print(f"Rentang data: dari {min_date} sampai {max_date}")
print(events_df.head())

In [ ]:
import pandas as pd

# Memuat data asli
events_df = pd.read_csv('/content/events.csv')
events_df['datetime'] = pd.to_datetime(events_df['timestamp'], unit='ms')

# Filter P1 (Training): 5/20/2015 hingga 7/20/2015
p1_start = '2015-05-20'
p1_end = '2015-07-20'
p1_df = events_df[(events_df['datetime'] >= p1_start) & (events_df['datetime'] <= p1_end)].copy()
p1_df['period'] = 'P1'

# Filter P2 (Testing): 7/29/2015 hingga 8/13/2015
p2_start = '2015-07-29'
p2_end = '2015-08-13'
p2_df = events_df[(events_df['datetime'] >= p2_start) & (events_df['datetime'] <= p2_end)].copy()
p2_df['period'] = 'P2'

# Menggabungkan data tersaring
filtered_events = pd.concat([p1_df, p2_df], ignore_index=True)

# Menyimpan hasil ke file CSV baru
filtered_events.to_csv('/content/filtered_events.csv', index=False)

print(f"Data P1 (Training): {len(p1_df)} baris")
print(f"Data P2 (Testing): {len(p2_df)} baris")
print(f"Total data tersaring: {len(filtered_events)} baris")
print("Sampel data tersaring:")
print(filtered_events.head())


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# Membaca data tersaring
print("Loading filtered dataset...")
df = pd.read_csv('/content/filtered_events.csv')

# Agar eksekusi visualisasi networkx cepat dan tidak overload di memori Colab,
# kita ambil subset acak dari interaksi untuk pemodelan graf lokal.
np.random.seed(42)
unique_users = df['visitorid'].unique()
sampled_users = np.random.choice(unique_users, size=1000, replace=False)
df_sampled = df[df['visitorid'].isin(sampled_users)].copy()

# --- 2.1 Nomor 3: Komposisi Subgraph ---
# Jenis simpul: Visitor (User) dan Item
num_users = df_sampled['visitorid'].nunique()
num_items = df_sampled['itemid'].nunique()

print("=== 2.1.3 Komposisi Subgraph ===")
print(f"Jumlah Simpul Visitor/User (V): {num_users}")
print(f"Jumlah Simpul Item (I): {num_items}")
print(f"Jumlah Total Simpul: {num_users + num_items}")
print(f"Jumlah Tepi Interaksi: {len(df_sampled)}")


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# --- 2.1 No 4 & 5: Vector Space Modelling (VSM) ---
# Membuat User-Item Interaction Matrix dari P1 (Training)
p1_events = df_sampled[df_sampled['period'] == 'P1']

# Memetakan ID simpul ke indeks matriks
user_list = sorted(p1_events['visitorid'].unique())
item_list = sorted(p1_events['itemid'].unique())

user_to_idx = {uid: i for i, uid in enumerate(user_list)}
item_to_idx = {iid: i for i, iid in enumerate(item_list)}

# Matriks interaksi (User x Item)
R = np.zeros((len(user_list), len(item_list)))
for _, row in p1_events.iterrows():
    u_idx = user_to_idx[row['visitorid']]
    i_idx = item_to_idx[row['itemid']]
    # Memberikan bobot sederhana berdasarkan tipe event (view=1, addtocart=2, transaction=3)
    weight = 1 if row['event'] == 'view' else (2 if row['event'] == 'addtocart' else 3)
    R[u_idx, i_idx] = max(R[u_idx, i_idx], weight)

print("=== 2.1.4 & 2.1.5 Vector Space Modelling (VSM) ===")
print(f"VSM User (Matriks R): Dimensi = {R.shape} (User x Item representation)")
print(f"VSM Item (Matriks R.T): Dimensi = {R.T.shape} (Item x User representation)")

# --- 2.1 No 6: Proyeksi Graf Manakah yang Lebih Baik? ---
# Proyeksi User-User Similarity Matrix
user_sim = cosine_similarity(R)
# Proyeksi Item-Item Similarity Matrix
item_sim = cosine_similarity(R.T)

print("\n=== 2.1.6 Analisis Proyeksi ===")
print("Proyeksi Item-Item umumnya lebih baik (efisien) karena perilaku preferensi item lebih stabil,")
print("sedangkan jumlah user aktif terus bertambah secara dinamis di dunia nyata.")

# --- 2.1 No 7 & 8: Similarity Graf (Sebelum vs Sesudah) ---
# Memilih satu acuan simpul User
ref_user_idx = 0
ref_user_id = user_list[ref_user_idx]
sim_scores = user_sim[ref_user_idx]

# Mendapatkan 5 record similarity tertinggi (mengecualikan diri sendiri)
top_5_indices = np.argsort(sim_scores)[::-1][1:6]
print(f"\n=== 2.1.7 (a) 5 User Paling Mirip dengan User Acuan {ref_user_id} ===")
for idx in top_5_indices:
    print(f"User ID: {user_list[idx]} | Similarity: {sim_scores[idx]:.4f}")

# Visualisasi Sebelum vs Sesudah Perhitungan Similarity (Networkx)
plt.figure(figsize=(12, 5))

# Graf Sebelum (Hanya bipartite interaksi user ke item untuk user acuan dan tetangganya)
G_before = nx.Graph()
for idx in list(top_5_indices) + [ref_user_idx]:
    uid = user_list[idx]
    for i_idx, val in enumerate(R[idx]):
        if val > 0:
            G_before.add_edge(f"U_{uid}", f"I_{item_list[i_idx]}")

plt.subplot(1, 2, 1)
nx.draw(G_before, with_labels=True, node_color='lightblue', font_size=8, node_size=500)
plt.title("Sebelum Similarity (Graf Bipartite)")

# Graf Sesudah (Proyeksi relasi similarity langsung antar User)
G_after = nx.Graph()
for idx in top_5_indices:
    if sim_scores[idx] > 0:
         G_after.add_edge(f"U_{ref_user_id}", f"U_{user_list[idx]}", weight=sim_scores[idx])

plt.subplot(1, 2, 2)
pos = nx.spring_layout(G_after)
nx.draw(G_after, pos, with_labels=True, node_color='salmon', font_size=8, node_size=500)
labels = nx.get_edge_attributes(G_after, 'weight')
labels = {k: f"{v:.2f}" for k, v in labels.items()}
nx.draw_networkx_edge_labels(G_after, pos, edge_labels=labels, font_size=8)
plt.title("Sesudah Similarity (Proyeksi User-User)")
plt.tight_layout()
plt.show()


In [ ]:
# --- 2.2 Analytics: Rekomendasi KNN & Evaluasi ---
print("=== 2.2.1 KNN Recommendations (k=5) ===")
# Memilih user acak secara reproducible
np.random.seed(42)
random_user_id = np.random.choice(user_list)
random_user_idx = user_to_idx[random_user_id]

# KNN (k=5)
k = 5
user_similarities = user_sim[random_user_idx]
nearest_neighbors = np.argsort(user_similarities)[::-1][1:k+1]

# Mengumpulkan item yang dikunjungi oleh KNN yang belum pernah dikunjungi oleh target user
target_visited = set(np.where(R[random_user_idx] > 0)[0])
recommendations = {}

for neighbor in nearest_neighbors:
    sim_w = user_similarities[neighbor]
    neighbor_items = np.where(R[neighbor] > 0)[0]
    for item in neighbor_items:
        if item not in target_visited:
            recommendations[item] = recommendations.get(item, 0) + sim_w * R[neighbor, item]

# Rekomendasi 3 item teratas
top_items_idx = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)[:3]
top_items_ids = [item_list[idx] for idx, score in top_items_idx]

print(f"Target User ID: {random_user_id}")
print(f"5 Tetangga Terdekat (KNN): {[user_list[n] for n in nearest_neighbors]}")
print(f"3 Item Rekomendasi Teratas: {top_items_ids}")

# --- 2.2.2 Evaluasi Metrik Hit@5 & NDCG@5 menggunakan Ground Truth dari P2 (Testing) ---
p2_events = df_sampled[df_sampled['period'] == 'P2']
target_p2_items = set(p2_events[p2_events['visitorid'] == random_user_id]['itemid'].unique())

print("\n=== 2.2.2 Evaluasi Metrik ===")
print(f"Item aktual yang dikunjungi oleh user pada periode P2 (Ground Truth): {list(target_p2_items)}")

# Menghitung Hit@5 (apakah ada item rekomendasi yang muncul di data aktual)
hits = [1 if item in target_p2_items else 0 for item in top_items_ids]
hit_at_5 = 1 if sum(hits) > 0 else 0

# Menghitung NDCG@5
idcg = sum([1.0 / np.log2(i + 2) for i in range(min(5, len(target_p2_items)))]) if len(target_p2_items) > 0 else 1.0
dcg = sum([hits[i] / np.log2(i + 2) for i in range(len(hits))])
ndcg_at_5 = dcg / idcg if idcg > 0 else 0.0

print(f"Hit@5: {hit_at_5}")
print(f"NDCG@5: {ndcg_at_5:.4f}")


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Membatasi visualisasi hanya untuk beberapa user agar graf terlihat rapi dan jelas
visual_users = user_list[:5]
sub_events = p1_events[p1_events['visitorid'].isin(visual_users)]

B = nx.Graph()

# Tambahkan simpul dengan atribut bipartite
B.add_nodes_from(visual_users, bipartite=0)
visual_items = sub_events['itemid'].unique()
B.add_nodes_from(visual_items, bipartite=1)

# Tambahkan tepi interaksi
for _, row in sub_events.iterrows():
    B.add_edge(row['visitorid'], row['itemid'])

# Atur posisi layout khusus untuk graf bipartite
pos = nx.bipartite_layout(B, visual_users)

plt.figure(figsize=(10, 6))
# Gambar simpul User
nx.draw_networkx_nodes(B, pos, nodelist=visual_users, node_color='skyblue', node_size=800, label='User')
# Gambar simpul Item
nx.draw_networkx_nodes(B, pos, nodelist=visual_items, node_color='lightgreen', node_size=600, label='Item')
# Gambar Tepi
nx.draw_networkx_edges(B, pos, width=1.5, edge_color='gray', alpha=0.7)
# Gambar Label
nx.draw_networkx_labels(B, pos, font_size=8, font_family='sans-serif')

plt.title("Visualisasi Struktur Jaringan Interaksi Bipartite (User - Item)", fontsize=14)
plt.legend(loc='upper right')
plt.axis('off')
plt.tight_layout()
plt.show()